# La Grulla Blanca — Definitivas Fuego + Metal + Agua
Benchmark mecánico pesado sobre `NEW_COMBAT_STATS_V0_1`. No es runtime ni cierre de balance.

Reconstruye desde GitHub el paquete consolidado **v0.2-followup-chains** y verifica SHA-256 antes de ejecutar.

Se prueban activación, cadenas posteriores, consumo de marcas/ventanas, clímax y expiraciones.

**Salida para devolver a ChatGPT:** `ULTI_3F_RESULTS_FOR_CHATGPT.zip`.


In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, urllib.parse, urllib.request, zipfile

OWNER = "Shein25"
REPO = "La-Grulla-Blanca"
BRANCH = "experiment/ultis-kaggle-3familias-v0.1"
ROOT_PATH = "experimentos/ultis/kaggle/package_chunks_v3"
EXPECTED_SHA256 = "2fbaafd40367f1104b6bd5a479601b1b64f09e0849d04700735e4618fcd52775"

def github_text(path):
    ref = urllib.parse.quote(BRANCH, safe="")
    path_q = "/".join(urllib.parse.quote(x, safe="") for x in path.split("/"))
    url = f"https://api.github.com/repos/{OWNER}/{REPO}/contents/{path_q}?ref={ref}"
    req = urllib.request.Request(url, headers={"Accept":"application/vnd.github+json","User-Agent":"La-Grulla-Blanca-Kaggle-LAB/1.2"})
    with urllib.request.urlopen(req, timeout=60) as r:
        obj = json.loads(r.read().decode("utf-8"))
    if obj.get("encoding") != "base64":
        raise RuntimeError(f"Contenido GitHub inesperado para {path}: {obj.get('encoding')}")
    return base64.b64decode(obj["content"]).decode("utf-8")

manifest = json.loads(github_text(f"{ROOT_PATH}/manifest.json"))
joined = "".join(github_text(f"{ROOT_PATH}/{name}").strip() for name in manifest["parts"])
if len(joined) != manifest["base64_length"]:
    raise RuntimeError(f"Base64 incompleto: {len(joined)} != {manifest['base64_length']}")
data = base64.b64decode(joined, validate=True)
sha = hashlib.sha256(data).hexdigest()
if len(data) != manifest["decoded_size_bytes"]:
    raise RuntimeError(f"Tamaño ZIP incorrecto: {len(data)} != {manifest['decoded_size_bytes']}")
if sha != EXPECTED_SHA256 or sha != manifest["sha256"]:
    raise RuntimeError(f"SHA-256 incorrecto: {sha}; esperado={EXPECTED_SHA256}; manifest={manifest['sha256']}")

ROOT = Path("/kaggle/working/ulti3f_pkg")
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)
zip_path = Path("/kaggle/working") / manifest["package"]
zip_path.write_bytes(data)
with zipfile.ZipFile(zip_path) as z:
    bad = z.testzip()
    if bad: raise RuntimeError(f"ZIP corrupto: {bad}")
    z.extractall(ROOT)
catalog=json.loads((ROOT/"ultimates_catalog_v0_1.json").read_text())
if not catalog.get("global_rules",{}).get("followup_chain_effects_are_tested"):
    raise RuntimeError("El paquete no activa la auditoría de cadenas posteriores")

print("GIT_PACKAGE_OK")
print("Revision:", manifest.get("revision"))
print("Branch:", BRANCH)
print("SHA-256:", sha)
print("Runner:", ROOT/"run_ultimates_3families_kaggle.py")


In [ ]:
# Smoke obligatorio antes de la corrida pesada.
SMOKE_OUT = Path('/kaggle/working/smoke_ulti_3f')
cmd = f"python {ROOT/'run_ultimates_3families_kaggle.py'} --mode smoke --runs 200 --workers 0 --output {SMOKE_OUT}"
print(cmd)
rc=os.system(cmd)
if rc != 0: raise RuntimeError(f"Smoke falló con código {rc}")
selfcheck=json.loads((SMOKE_OUT/'SELF_CHECK.json').read_text())
if not selfcheck.get('pass'): raise RuntimeError(f"SELF_CHECK falló: {selfcheck}")
print('SMOKE + SELF_CHECK: PASS')


In [ ]:
# Corrida pesada: 45 escenarios x 100.000 corridas.
RUNS = 100_000
OUT = Path('/kaggle/working/results_ulti_3f')
cmd = f"python {ROOT/'run_ultimates_3families_kaggle.py'} --mode heavy --runs {RUNS} --workers 0 --output {OUT}"
print(cmd)
rc=os.system(cmd)
if rc != 0: raise RuntimeError(f'Runner terminó con código {rc}')


In [ ]:
from IPython.display import FileLink, display
result=OUT/'ULTI_3F_RESULTS_FOR_CHATGPT.zip'
print('Resultado compacto:', result, f'{result.stat().st_size/1024:.1f} KiB')
display(FileLink(str(result)))
print('Devuelve únicamente ese ZIP a ChatGPT.')
